<a href="https://colab.research.google.com/github/Siva-S-05/webscraping/blob/main/websccrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
data = [] # Ensure data list is re-initialized or clear before populating

item_containers = soup.find_all('li', class_='s-card')

print(f"Found {len(item_containers)} item containers.")

for item in item_containers:
    title = None
    link = None
    price = None

    # Extract Link - from the <a> tag with image
    title_link_element = item.find('a', class_='s-card__link')
    if title_link_element:
        link = title_link_element.get('href')

    # Extract Title - it's a separate div, NOT inside the link
    title_tag = item.find('div', class_='s-card__title')
    if title_tag:
        title = title_tag.text.strip()

    # Extract Price
    price_element = item.find('span', class_='s-card__price')
    if price_element:
        price = price_element.text.strip()

    # Only add items that have at least a title and a link
    # Skip ads like "Shop on eBay"
    if title and link and title != "Shop on eBay":
        data.append({
            'Title': title,
            'Price': price,
            'Link': link
        })

print(f"Extracted {len(data)} items.")
# Print first 5 extracted items for inspection
for i, item in enumerate(data[:5]):
    print(f"Item {i+1}: {item}")

Found 62 item containers.
Extracted 60 items.
Item 1: {'Title': 'Dell Latitude Laptop Computer PC Intel i7 Up To 32GB RAM 1TB SSD Windows 11Opens in a new window or tab', 'Price': '$263.73', 'Link': 'https://www.ebay.com/itm/286393092388?_skw=laptop&itmmeta=01KEHHQ9ATM3DYJ0478VEDEET9&hash=item42ae5bc924:g:L-wAAeSwATBpW-jQ&itmprp=enc%3AAQAKAAAAwFkggFvd1GGDu0w3yXCmi1d6aXcsBLilrZYpd3zKnlKiMXKNiOWLDqpLowDHMdXRrN%2BykYLyqgs4n%2Fdlj850IX%2BI33cgDxbyKRQVXxELok3YtTjAydFU9NQnWeQVAYPmqbD7r73q6DMVekfuhdMpkqQOkxTc7Q3R%2FWMNC0ZK62RtoPND1MYVvZHAmJwdyK5PcP%2FfuU2ER7YS3Y5YYFsbqeRA840aLVod8U592MAdEYspgACwI%2BYHPVI%2FN8C2oONJrw%3D%3D%7Ctkp%3ABlBMUIKW3bH0Zg'}
Item 2: {'Title': 'HP Laptop 14" HD AMD Ryzen 3 3200U 4GB RAM 128GB SSD Windows 11 Pro PCOpens in a new window or tab', 'Price': '$125.30', 'Link': 'https://www.ebay.com/itm/266791101115?_skw=laptop&itmmeta=01KEHHQ9AT8DF6CEZCP7M99TEK&hash=item3e1dfd22bb:g:OqQAAeSwzk9pYJjQ&itmprp=enc%3AAQAKAAAAwFkggFvd1GGDu0w3yXCmi1e9szlWaw%2FpyUq7I60%2BZz7HNKTd%2Bd2

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

class EbayScraper:
    """
    A web scraper for eBay product listings.
    Extracts title, price, link, condition, and shipping info.
    """

    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        self.data = []

    def scrape_ebay(self, search_query, pages=1):
        """
        Scrape eBay search results for a given query.

        Args:
            search_query (str): The search term (e.g., 'laptop', 'iphone')
            pages (int): Number of pages to scrape
        """
        print(f"Starting scrape for: {search_query}")
        print(f"Pages to scrape: {pages}")

        for page in range(1, pages + 1):
            # eBay pagination: _pgn parameter
            url = f"https://www.ebay.com/sch/i.html?_nkw={search_query}&_pgn={page}"

            print(f"\nScraping page {page}...")
            print(f"URL: {url}")

            try:
                response = requests.get(url, headers=self.headers)
                response.raise_for_status()

                soup = BeautifulSoup(response.content, 'html.parser')

                # Find all product listing containers
                item_containers = soup.find_all('li', class_='s-card')

                print(f"Found {len(item_containers)} items on page {page}")

                for item in item_containers:
                    item_data = self.extract_item_data(item)

                    # Only add valid items (skip ads)
                    if item_data and item_data['title'] != "Shop on eBay":
                        self.data.append(item_data)

                # Be respectful - add delay between requests
                time.sleep(2)

            except requests.exceptions.RequestException as e:
                print(f"Error fetching page {page}: {e}")
                continue

        print(f"\nTotal items extracted: {len(self.data)}")
        return self.data

    def extract_item_data(self, item):
        """
        Extract individual item data from HTML element.

        Args:
            item: BeautifulSoup element containing product data

        Returns:
            dict: Product information
        """
        try:
            # Extract Title
            title_tag = item.find('div', class_='s-card__title')
            title = title_tag.text.strip() if title_tag else None

            # Extract Link
            link_tag = item.find('a', class_='s-card__link')
            link = link_tag.get('href') if link_tag else None

            # Extract Price
            price_tag = item.find('span', class_='s-card__price')
            price = price_tag.text.strip() if price_tag else None

            # Extract Condition (if available)
            condition_tag = item.find('span', class_='SECONDARY_INFO')
            condition = condition_tag.text.strip() if condition_tag else "Not specified"

            # Extract Shipping Info
            shipping_tag = item.find('span', class_='s-item__shipping')
            shipping = shipping_tag.text.strip() if shipping_tag else "See details"

            # Only return if we have at least title and link
            if title and link:
                return {
                    'title': title,
                    'price': price,
                    'condition': condition,
                    'shipping': shipping,
                    'link': link
                }

            return None

        except Exception as e:
            print(f"Error extracting item: {e}")
            return None

    def save_to_csv(self, filename='ebay_data.csv'):
        """
        Save scraped data to CSV file.

        Args:
            filename (str): Output CSV filename
        """
        if not self.data:
            print("No data to save!")
            return

        df = pd.DataFrame(self.data)
        df.to_csv(filename, index=False, encoding='utf-8')
        print(f"\nData saved to {filename}")
        print(f"Total records: {len(df)}")

        # Display summary
        print("\nDataset Summary:")
        print(df.head())
        print(f"\nColumns: {list(df.columns)}")
        print(f"Shape: {df.shape}")

        return df

    def clean_price(self, price_str):
        """
        Clean price string and convert to float.

        Args:
            price_str (str): Price string like "$123.45" or "$1,234.56"

        Returns:
            float: Cleaned price value
        """
        if not price_str:
            return None

        # Remove currency symbols and commas
        cleaned = re.sub(r'[^\d.]', '', price_str)

        try:
            return float(cleaned)
        except ValueError:
            return None

    def get_price_statistics(self):
        """
        Calculate price statistics from scraped data.
        """
        if not self.data:
            print("No data available!")
            return

        df = pd.DataFrame(self.data)

        # Clean prices
        df['price_numeric'] = df['price'].apply(self.clean_price)

        # Remove None values
        prices = df['price_numeric'].dropna()

        if len(prices) == 0:
            print("No valid prices found!")
            return

        print("\nPrice Statistics:")
        print(f"Average Price: ${prices.mean():.2f}")
        print(f"Median Price: ${prices.median():.2f}")
        print(f"Min Price: ${prices.min():.2f}")
        print(f"Max Price: ${prices.max():.2f}")
        print(f"Total items with prices: {len(prices)}")


# Example Usage
if __name__ == "__main__":
    # Create scraper instance
    scraper = EbayScraper()

    # Scrape eBay for laptops (2 pages)
    scraper.scrape_ebay(search_query='laptop', pages=2)

    # Save to CSV
    df = scraper.save_to_csv('ebay_laptops.csv')

    # Get price statistics
    scraper.get_price_statistics()

    # Example: Scrape different product
    # scraper2 = EbayScraper()
    # scraper2.scrape_ebay(search_query='iphone 13', pages=3)
    # scraper2.save_to_csv('ebay_iphones.csv')

Starting scrape for: laptop
Pages to scrape: 2

Scraping page 1...
URL: https://www.ebay.com/sch/i.html?_nkw=laptop&_pgn=1
Found 62 items on page 1

Scraping page 2...
URL: https://www.ebay.com/sch/i.html?_nkw=laptop&_pgn=2
Found 0 items on page 2

Total items extracted: 60

Data saved to ebay_laptops.csv
Total records: 60

Dataset Summary:
                                               title    price      condition  \
0  Dell Latitude Laptop Computer PC Intel i7 Up T...  $263.73  Not specified   
1  Dell Latitude 14" Laptop Computer Intel Up To ...  $210.98  Not specified   
2  Dell Latitude 3190 2-in-1 Touch Laptop Intel P...  $113.41  Not specified   
3  HP Laptop 14" HD AMD Ryzen 3 3200U 4GB RAM 128...  $125.30  Not specified   
4  Acer Nitro V 16" Laptop R5 16GB RAM 512GB SSD ...  $585.89  Not specified   

      shipping                                               link  
0  See details  https://www.ebay.com/itm/286393092388?_skw=lap...  
1  See details  https://www.ebay.com/itm